# Les pénuries de carburant se propagent-elles comme une épidémie ?
### La crise d'octobre 2022 dans les stations-service françaises

Auteurs : Sarah LIABES, Adam Trabelsi, Eden Szezot

## Sommaire

- Introduction
- I. Données
  - A. Ruptures de carburant par station
  - B. Raffineries, dépôts et dates de grève
  - C. Sources de croisement (chiffres officiels, INSEE, contours)
  - D. Contrôles qualité et export
- II. Dynamique de la crise
- III. Choc commun ou contagion locale ?
  - A. H1 : diffusion depuis les raffineries et dépôts en grève
  - B. H2 : report des automobilistes sur les stations voisines
- IV. Comparaison avec une vraie épidémie : le COVID-19
- Conclusion et limites

## Introduction

**Contexte.** 

**Problématique.** Pendant la crise d'octobre 2022, les ruptures en stations-service s'expliquent-elles par un **choc logistique commun** (raffineries et dépôts en grève) ou par une **contagion locale**, les automobilistes se reportant sur les stations voisines ?

**Hypothèses.**
- **H1 — choc commun :** les ruptures se concentrent autour des sites en grève et les stations d'une même zone tombent en même temps.
- **H2 — contagion par report :** une station a plus de chances de tomber en rupture lorsque ses voisines le sont déjà, à approvisionnement comparable.

**Démarche.** 

### Configuration

Le code est rangé dans le package `src/penuries/`. Les cellules ci-dessous l'importent ; les données sont téléchargées depuis leurs sources officielles.

In [1]:
import sys

sys.path.append("src")

import pandas as pd

from penuries import collecte, nettoyage, indicateurs, visualisation

## I. Données

### A. Ruptures de carburant par station

_Source : flux annuel 2022 du site prix-carburants.gouv.fr (données ouvertes du ministère de l'Économie)._

Le flux annuel 2022 recense, pour chaque station-service, ses prix et ses ruptures (carburant, date de début, date de fin si elle est connue). Nous le téléchargeons directement depuis la source officielle, puis nous en extrayons deux tables :
- `df_stations` : une ligne par station (coordonnées, code postal, autoroute ou route, nombre de mises à jour de prix sur la période) ;
- `df_ruptures` : une ligne par épisode de rupture.



In [2]:
chemin_xml = collecte.telecharger_flux_annuel(2022)

df_stations = collecte.lire_stations(chemin_xml)
df_ruptures = collecte.lire_ruptures(chemin_xml)

print(f"{len(df_stations):,} stations, dont {(df_stations['nb_maj_periode'] > 0).sum():,} actives entre septembre et novembre 2022")
print(f"{len(df_ruptures):,} épisodes de rupture au total")
df_ruptures.head()

13,645 stations, dont 9,563 actives entre septembre et novembre 2022
195,032 épisodes de rupture au total


,id_station,latitude,longitude,code_postal,type_route,carburant,debut,fin
0,1000001,4620100,519800,01000,R,GPLc,2017-09-16 09:50:23,NaT
1,1000001,4620100,519800,01000,R,E10,2018-12-13 09:49:49,NaT
2,1000001,4620100,519800,01000,R,E85,2017-09-16 09:50:23,NaT
3,1000001,4620100,519800,01000,R,SP98,2022-10-21 07:51:17,2022-10-21 07:52:49
4,1000001,4620100,519800,01000,R,SP95,2022-10-11 08:31:05,2022-10-11 17:10:56


In [3]:
df_ruptures["debut"].dt.to_period("M").value_counts().sort_index().loc["2022-06":"2022-12"]

debut
2022-06     1654
2022-07     2927
2022-08     2461
2022-09    24528
2022-10    73716
2022-11    36292
2022-12    16000
Freq: M, Name: count, dtype: int64

Par exemple, les ruptures commencées en octobre 2022 sont environ 30 fois plus nombreuses qu'en août, ce qui confirme l'ampleur de la crise. Le fichier contient aussi des ruptures très anciennes sans date de fin (carburants que la station ne vend plus) : nous les traitons dans la partie D.

### B. Raffineries, dépôts et dates de grève

_Table construite à la main (`data/manuel/`), une source citée par ligne._

### C. Sources de croisement

_Chiffres officiels quotidiens du ministère de la Transition énergétique, population INSEE par département, contours des départements._

### D. Contrôles qualité et export

_Valeurs manquantes, doublons, cohérence des dates, comparaison avec les chiffres officiels. Export Parquet sur le stockage S3 du SSP Cloud._